# Notebook 26 — Omni-Judge on the SAME gate. Gate only.

`KbsdJames/Omni-Judge` — Llama-3.1-8B-Instruct instruction-tuned on GPT-4o
evaluation data. The second open-judge candidate, after LiveMath-Judge failed
this gate.

**This notebook runs the gate and stops.** No 300-item run, no diagnostic
sample, no accuracy. Standing rule: gate first → if it fails, record the
failure and stop.

**The bar is identical to LiveMath-Judge's**, deliberately — comparing two
judges against two different gates would prove nothing.

| probe | what it tests | LiveMath-Judge |
|---|---|---|
| item 55 | reject a mathematically corrected but non-faithful answer | ✅ passed |
| item 273 | reject a **non-answer** (coin-tossing setup prose) | ❌ **failed** |

**Three differences from LiveMath-Judge that shape this notebook:**

1. **It emits a justification.** Output is structured — answer, judgement
   TRUE/FALSE, justification — where LiveMath-Judge gave a bare `\boxed{yes}`.
   So the "does it solve the maths rather than compare?" question is finally
   answerable, and `looks_like_solving` has something to read.
2. **The prompt is baked into `tokenizer.get_context()`** — there is no
   supported way to add a fidelity clause. Run native and gate it. The clause
   changed 2 of 40 items on LiveMath-Judge and both toward *leniency*, so
   inventing an unsupported call path to inject one is not worth it.
3. **Custom tokenizer methods need `trust_remote_code=True`.** That is the
   shape that broke InternVL3 here (`all_tied_weights_keys` under transformers
   v5). Lower risk — custom tokenizer, not a custom model class — but if it
   fails that way it is a known failure mode, not worth debugging on paid GPU.

If Omni-Judge also accepts item 273, the conclusion is that **open math judges
are not reliable fidelity scorers for perturbed-answer transcription** — a
Limitations finding, and the point to stop testing free judges.

In [1]:
# Auth + code access. GPU: Omni-Judge is 8B (Llama-3.1).
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"
OUT_DIR = f"{PROJECT_DIR}/audit"
os.makedirs(OUT_DIR, exist_ok=True)

with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.judge

assert pilot.canonicalize.latex_parser_available(), "SymPy LaTeX parser broken"
import torch
assert torch.cuda.is_available(), "no GPU: enable a GPU runtime"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"gate items: {pilot.judge.GATE_ITEMS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
GPU: NVIDIA A100-SXM4-40GB
gate items: (55, 273)


In [2]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

run = pd.read_csv(f"{RESULTS_DIR}/scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv")
assert len(run) == 300

MID = pilot.judge.OMNI_MODEL_ID
# trust_remote_code is required for get_context/parse_response. If this raises
# an all_tied_weights_keys AttributeError it is the known transformers-v5
# custom-code breakage that killed InternVL3 here -- stop rather than debug.
# TWO tokenizers, and the reason matters. The custom OmniJudgeTokenizer
# supplies get_context/parse_response, but its DECODE is broken under
# transformers v5: it returns space-separated tokens with the byte-BPE marker
# intact -- "E quiv ale Ġn ce Jud gment F AL ĠS ĠE" instead of
# "## Equivalence Judgment FALSE". parse_response then finds none of its
# markers and returns all-None, which reads as the judge failing when it is
# the DECODER failing. The plain fast tokenizer decodes the same vocab
# correctly, so use it for text and the custom one only for its two methods.
tokenizer = AutoTokenizer.from_pretrained(MID, trust_remote_code=True, token=HF_TOKEN)
plain_tok = AutoTokenizer.from_pretrained(MID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MID, torch_dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN).eval()

for m in ("get_context", "parse_response"):
    assert hasattr(tokenizer, m), (
        f"tokenizer has no {m}() -- the custom code did not load, so the "
        "official call path is unavailable. Do not hand-build a prompt as a "
        "substitute; that would be a different experiment.")
print(f"loaded {MID}; custom tokenizer methods present")

terminators = [tokenizer.eos_token_id]
eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if isinstance(eot, int) and eot >= 0:
    terminators.append(eot)


def omni_judge(question, gold, answer):
    """(verdict, raw) using the model's OWN prompt builder and parser."""
    ctx = tokenizer.get_context(question or "", gold or "", answer or "")
    enc = tokenizer(ctx, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
            do_sample=False, max_new_tokens=pilot.judge.OMNI_MAX_NEW_TOKENS,
            eos_token_id=terminators,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    new = out[0][enc["input_ids"].shape[1]:].cpu().tolist()
    text = plain_tok.decode(new, skip_special_tokens=True)
    if pilot.judge.looks_bpe_mangled(text):
        # Fall back rather than silently judging on garbage.
        text = pilot.judge.demangle_bpe(tokenizer.decode(new, skip_special_tokens=True))
    try:
        parsed = tokenizer.parse_response(text)
        judgement = parsed.get("judgement") if isinstance(parsed, dict) else None
    except Exception as e:                       # keep the raw text either way
        judgement, parsed = None, f"<parse_response failed: {e}>"
    return pilot.judge.parse_omni_judgement(judgement), f"{parsed}\n---RAW---\n{text}"


print("backend ready (native prompt, greedy)")

config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenization_omnijudge.py:   0%|          | 0.00/6.76k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/KbsdJames/Omni-Judge:
- tokenization_omnijudge.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

loaded KbsdJames/Omni-Judge; custom tokenizer methods present
backend ready (native prompt, greedy)


In [3]:
# THE GATE. Identical bar to LiveMath-Judge: both items must come back
# `incorrect`. Then STOP -- no 300-item run either way.
gate = pilot.judge.run_gate_with(omni_judge, run, label="Omni-Judge")

# A verdict read off a mangled transcript is not a verdict. Check before
# believing anything below.
mangled = {i: pilot.judge.looks_bpe_mangled(r) for i, r in gate["raw"].items()}
assert not any(mangled.values()), (
    f"DECODE STILL BROKEN on {[i for i, m in mangled.items() if m]} -- the "
    "output is raw BPE tokens, so `unclear` here means the DECODER failed, "
    "not the judge. Fix the decode before recording any gate result.")
print("decode OK on both probes\n")

print(f"Omni-Judge gate: {'PASSED' if gate['passed'] else 'FAILED'}\n")
for i, v in gate["verdicts"].items():
    what = ("reject a corrected-but-non-faithful answer" if i == 55
            else "reject a NON-ANSWER (setup prose)")
    print(f"item {i} — {what}")
    print(f"   verdict: {v}   {'OK' if v == 'incorrect' else '<-- FAILS'}")
    print(f"   appears to solve rather than compare: {gate['solving'][i]}")
    print(f"   raw:\n{str(gate['raw'][i])[:700]}\n")

lines = [
    "# Omni-Judge — gate only\n",
    f"Model: `{pilot.judge.OMNI_MODEL_ID}` · native prompt · 2026-08-12\n",
    f"**{'PASSED' if gate['passed'] else 'FAILED'}** — same gate as "
    "LiveMath-Judge (items 55 and 273, both must be `incorrect`).\n",
    "| item | tests | verdict | appears to solve |", "|---|---|---|---|",
]
for i, v in gate["verdicts"].items():
    what = ("corrected-but-non-faithful" if i == 55 else "non-answer")
    lines.append(f"| {i} | {what} | `{v}` | {gate['solving'][i]} |")
if not gate["passed"]:
    lines += [f"\n> {gate['note']}\n",
              "\nWith LiveMath-Judge having failed the same gate, the "
              "conclusion is that **open math judges are not reliable "
              "fidelity scorers for perturbed-answer transcription**. Stop "
              "testing free judges.\n"]
else:
    lines += ["\nPassing the gate is a NECESSARY condition, not a sufficient "
              "one. It licenses a diagnostic sample next — not a scoring "
              "run.\n"]
lines.append("\n## Raw output\n")
for i, r in gate["raw"].items():
    lines.append(f"**item {i}**\n\n```\n{str(r)[:1200]}\n```\n")

MD_PATH = f"{OUT_DIR}/omni_judge_gate_20260812.md"
pilot.judge.pathlib_write(MD_PATH, "\n".join(lines))
print(f"summary -> {MD_PATH}")
print("\nSTOP HERE. No 300-item run in this notebook, by design.")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer OmniJudgeTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Omni-Judge gate: FAILED

item 55 — reject a corrected-but-non-faithful answer
   verdict: unclear   <-- FAILS
   appears to solve rather than compare: False
   raw:
{'answer': None, 'judgement': None, 'justification': None}
---RAW---
\ frac { tan x + tan Ġy }{ 1 -t an x Ġtan Ġy } # Ġ# E quiv ale Ġn ce Jud gment F AL ĠS ĠE # Ġ# J us Ġt ification The Ġstudent 's Ġanswer Ġis Ġincorrect Ġin Ġthe Ġcontext Ġof Ġthe Ġproblem , Ġwhich Ġasks Ġto Ġprove Ġthe Ġidentity Ġ\ [\ tan (x +y )= \ frac {\ tan Ġx +\ tan Ġy }{ 1 +\ tan Ġx Ġ\ tan Ġy } .\ ] ĠThe Ġreference Ġanswer Ġis Ġ\ [\ tan (x +y )= \ text { red }{ \ frac {\ tan Ġx +\ tan Ġy }{ 1 +\ tan Ġx Ġ\ tan Ġy }} .\ ] ĠThe Ġstudent 's Ġsolution , Ġhowever , Ġprovides Ġthe Ġexpression Ġ\ [\ tan (x +y )= \ frac {\ tan Ġx +\ tan Ġy }{ 1 -\ tan Ġx Ġ\ tan Ġy } .\ ] ĠThe Ġdiscrepancy Ġin Ġthe Ġsigns Ġin Ġthe Ġdenominator 

item 273 — reject a NON-ANSWER (setup prose)
   verdict: unclear   <-- FAILS
   appears to solve rather than compare: False
   raw:
{